In [3]:
import os
os.chdir("/Users/yesicarb/Desktop/UIE/3º Curso/2 SEM/PROYECTO/emotion/multimodal_emotion")

import pandas as pd
import numpy as np
import sys
sys.path.append("src")
from cv_classic.feature_extractor import extract_features

df = pd.read_csv("data/processed/labels.csv")

print("Re-extrayendo features...")
features = []
for i, (_, row) in enumerate(df.iterrows()):
    features.append(extract_features(row['image_path']))
    if i % 500 == 0:
        print(f"  {i}/{len(df)}")

X = np.array(features)
print(f"Features listas: {X.shape}")

Re-extrayendo features...
  0/4869
  500/4869
  1000/4869
  1500/4869
  2000/4869
  2500/4869
  3000/4869
  3500/4869
  4000/4869
  4500/4869
Features listas: (4869, 114)


In [3]:
svm, scaler, le, y_test, probas = run_cv_classic(df)

Extrayendo features de imagen...
  0/4869 imágenes procesadas
  500/4869 imágenes procesadas
  1000/4869 imágenes procesadas
  1500/4869 imágenes procesadas
  2000/4869 imágenes procesadas
  2500/4869 imágenes procesadas
  3000/4869 imágenes procesadas
  3500/4869 imágenes procesadas
  4000/4869 imágenes procesadas
  4500/4869 imágenes procesadas
Entrenando SVM...

=== CV Clásico (SVM) ===
              precision    recall  f1-score   support

    negative       0.34      0.28      0.31       244
     neutral       0.41      0.46      0.43       384
    positive       0.39      0.39      0.39       346

    accuracy                           0.39       974
   macro avg       0.38      0.38      0.38       974
weighted avg       0.39      0.39      0.39       974

Resultados guardados en results/metrics_cv_classic.json


In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
import numpy as np, json

# Reutilizar las features ya extraídas
le = LabelEncoder()
y  = le.fit_transform(df['label'].tolist())

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print("Grid Search — SVM...")
param_grid_svm = {
    'C':      [0.1, 1, 10, 100],
    'gamma':  ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf', 'linear']
}

gs_svm = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid_svm,
    cv=5, scoring='f1_macro',
    n_jobs=-1, verbose=1
)
gs_svm.fit(X_tr_sc, y_train)

print(f"\nMejores parámetros SVM: {gs_svm.best_params_}")
print(f"Mejor F1 CV:            {gs_svm.best_score_:.4f}")

y_pred_svm = gs_svm.predict(X_te_sc)
probas_svm = gs_svm.predict_proba(X_te_sc)

print("\n=== CV Clásico SVM Optimizado ===")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))
print(f"F1 macro test: {f1_score(y_test, y_pred_svm, average='macro'):.4f}")

# Guardar
results_cv_opt = {
    'f1_macro':    f1_score(y_test, y_pred_svm, average='macro'),
    'best_params': gs_svm.best_params_,
    'probas':      probas_svm.tolist()
}
with open('results/metrics_cv_classic.json', 'w') as f:
    json.dump(results_cv_opt, f, indent=2)
print("Guardado en results/metrics_cv_classic.json")

Grid Search — SVM...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Mejores parámetros SVM: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Mejor F1 CV:            0.3768

=== CV Clásico SVM Optimizado ===
              precision    recall  f1-score   support

    negative       0.34      0.28      0.31       244
     neutral       0.41      0.46      0.43       384
    positive       0.39      0.39      0.39       346

    accuracy                           0.39       974
   macro avg       0.38      0.38      0.38       974
weighted avg       0.39      0.39      0.39       974

F1 macro test: 0.3761
Guardado en results/metrics_cv_classic.json
